### Creacion de credenciales.

with open('postgres_password.txt', 'r', encoding='utf-8') as f:
    postgres_password = f.read().strip()   ----> Cotranseña en un txt con git.ignore

    
POSTGRES_CONFIG = {
    'host': '127.0.0.1',
    'database': 'FleetLogix',
    'user': 'postgres',
    'password': postgres_password,
    'port': 5432
}



## Snowflake Credenciaels
    Se usaron los siguiente comandos para crear las constraseñas, privada y publica. Las cuales estan excluidas del Proyecto por un git ignore.
openssl genrsa 2048 | openssl pkcs8 -topk8 -v1 PBE-SHA1-2DES -nocrypt -out rsa_key.p8
openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub

-- 1. Crear el rol de ETL

CREATE ROLE IF NOT EXISTS ETL_ROLE;    ---->
Creamos el ETL_ROLE para otorgar únicamente los permisos necesarios de escritura y procesamiento, protegiendo así la seguridad de tu cuenta principal. 

Luego el usuario test02 actúa como una "cuenta de servicio" independiente, permitiendo que el script se conecte de forma automática sin depender de tu acceso personal. Esta separación garantiza que el sistema sea auditable, escalable y ciberseguro al usar llaves RSA en lugar de contraseñas tradicionales. Al final, esta estructura profesional evita que un error en el código pueda comprometer o borrar toda tu base de datos.


-- 2. Asignar permisos al Warehous

GRANT USAGE ON WAREHOUSE FLEETLOGIX_WH TO ROLE ETL_ROLE;

-- 3. Asignar permisos a la Base de Datos y Esquema
GRANT ALL PRIVILEGES ON DATABASE FLEETLOGIX_DW TO ROLE ETL_ROLE;    
GRANT ALL PRIVILEGES ON SCHEMA FLEETLOGIX_DW.ANALYTICS TO ROLE ETL_ROLE;
USE ROLE ACCOUNTADMIN; -- O un rol con permisos altos
GRANT ALL PRIVILEGES ON SCHEMA FLEETLOGIX_DW.ANALYTICS TO ROLE ETL_ROLE;
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA FLEETLOGIX_DW.ANALYTICS TO ROLE ETL_ROLE;

-- 4. Crear el usuario para el proceso automático
CREATE USER "test02"
PASSWORD = 'Henryptft05'
DEFAULT_ROLE = ETL_ROLE,
DEFAULT_WAREHOUSE = FLEETLOGIX_WH,
MUST_CHANGE_PASSWORD = FALSE;

-- 5. Vincular el rol al usuario
GRANT ROLE ETL_ROLE TO USER "test02";

-- 6. Configurar la llave pública 
ALTER USER "test02" SET RSA_PUBLIC_KEY= :)

describe user "test02";

## Python credenciales -> snowflake
SNOWFLAKE_CONFIG = {
    'user': 'TEST02',
    'account': 'KDWMDJQ-SN80270',  
    'password': 'Henryptft05', 
    'warehouse': 'FLEETLOGIX_WH',
    'database': 'FLEETLOGIX_DW',
    'schema': 'ANALYTICS',
    'private_key_path': 'rsa_key.p8'
}



## def extract_daily_data() --> To Do
Crear una Query que recupere las entregas realizadas el día anterior desde PostgreSQL. 

Para ello se uni las tablas deliveries, trips y routes, obteniendo tanto los datos operativos de la entrega como la información del viaje y de la ruta. El filtro se aplica sobre delivered_datetime, considerando únicamente registros con estado delivered, ya que son los que permiten calcular métricas completas en la etapa de transformación.

La query ahora:

-une deliveries, trips y routes
-trae las columnas que necesita transform_data()
-filtra solo entregas con estado delivered
-toma únicamente las del día anterior usando delivered_datetime




 ## def _calculate_daily_totals(self):

 # creación de tabla

Se implementó la creación de la tabla daily_totals en Snowflake. Esta tabla funciona como una estructura de resumen diario pensada para consultas analíticas y reportes rápidos.
La tabla almacena la fecha resumida, el identificador del lote ETL (etl_batch_id) y métricas agregadas como cantidad total de entregas, entregas a tiempo, entregas demoradas, peso total transportado, distancia total recorrida, combustible consumido, costo total e ingresos estimados. También se agregó un campo de timestamp para registrar el momento de creación del resumen.


# inserción de agregados

También se completó la consulta de inserción que carga en daily_totals los datos agregados del batch actual. Esta operación toma como fuente la tabla fact_deliveries, que ya contiene las entregas transformadas y cargadas en el modelo analítico.

A partir de esos registros se agrupan los datos por fecha y por lote ETL, calculando indicadores diarios. Entre ellos se incluyen el total de entregas procesadas, cuántas fueron realizadas a tiempo, cuántas presentaron demora, el peso total movilizado, la distancia acumulada, el combustible total consumido, el costo estimado total y el ingreso estimado total.

De esta manera, el pipeline no solo carga el detalle de las entregas en la tabla de hechos, sino que además deja preparado un resumen diario que puede ser usado directamente para visualización o análisis posterior.

## EJECUCION

En la funcion  EXTRACT_DAYLY(), se modifico el intervalo de fecha de datos extraido de el dia anterior a 60 dias previos a fines de extraer una buena cantidad de datos para las prubas.

  AND DATE(d.delivered_datetime) >= CURRENT_DATE - INTERVAL '60 day'


# Error en la ejecucion
Durante la ejecución del pipeline, la extracción de datos se realizó correctamente y recuperó 9986 registros. Sin embargo, la transformación falló por una fecha generada dentro del propio código (9999-12-31) para representar una vigencia abierta. El problema no proviene del dataset fuente, sino de una limitación de pandas, que no soporta ese valor dentro de su rango de timestamps. Por lo tanto, no se trata de suciedad en los datos, sino de un ajuste técnico necesario en la implementación del ETL.
 
 Error en ---> df['valid_to'] = pd.to_datetime('9999-12-31')
 soulucion ---> df['valid_to'] = pd.to_datetime('2099-12-31')



# Error en la ejecucion
Luego de corregir la etapa de transformación, el pipeline logró procesar 5544 registros y cargar correctamente la tabla de hechos. Sin embargo, se detectó un error en la carga de la dimensión dim_customer. El problema no estuvo en los datos extraídos, sino en la estructura de inserción utilizada en el MERGE, ya que la tabla exige un valor para customer_key y la sentencia no lo estaba generando ni insertando. En consecuencia, la carga de dimensiones falló parcialmente, aunque el resto del flujo ETL pudo continuar.

# Error en la ejecucion
Durante las pruebas del pipeline se detectó un problema de rendimiento cuando la consulta de extracción se configuró para recuperar registros de los últimos 30 o 60 días. En ese escenario, la cantidad de entregas extraídas aumentaba considerablemente y, aunque la extracción y la transformación seguían ejecutándose relativamente rápido, la etapa de carga de dimensiones comenzaba a demorarse de forma excesiva.


Por tal motivo modifique el rango de extraccion a un dia especifico (10 de febrero) que fue el ultimo dia que se cargaron datos para  poder realizar las prubas de funcionamiento del pipeline.

 AND DATE(d.delivered_datetime) = '2026-02-10'

## Ejecucion Satisfactoria:
c:\Users\not rules\OneDrive\Desktop\ProyctoIntegrador2\A3-05_etl_pipeline_estudiantes.py:109: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


  df = pd.read_sql(query, self.pg_conn)

2026-03-25 00:16:04,667 - INFO -  Totales diarios calculados

2026-03-25 00:16:04,789 - INFO -  Conexiones cerradas

2026-03-25 00:16:04,789 - INFO -  ETL completado en 13.32 segundos

2026-03-25 00:16:04,789 - INFO -  Métricas: {

    "records_extracted": 19,

      "records_transformed": 11,

        "records_loaded": 11,
        
          "errors": 0
          
          
          }


Asi mismo se detecta que el PIPELINE otorgado no esta cargando realmente:

DIM_ROUTE
DIM_DATE
DIM_TIME
DIM_VEHICLE
DIM_DRIVER
STAGING_DAILY_LOAD
 
Solamente tiene la capacidad efectica de cargar las tablas DIM_CUSTOMER /FACT_DELIVERIES /DAILY_TOTALS


## Se adjunta en el "screenSnowFlake.doc" 
Screen shots de lo documentado en la plataforma Snowflake y los resultados obtenidos.
